# March 18 `other`-Pool EB Triage

Deterministic, read-only notebook for ranking likely eclipsing binaries inside the reviewed March 18 `other` pool.
It uses only candidate stats already present in the March 18 candidate table and does not rerun period searches or ML.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from malca.review.other_eb_triage import (
    BIN_ORDER,
    DEFAULT_CANDIDATES_SOURCE,
    DEFAULT_EVENT_CLASS_FILTER,
    DEFAULT_EXPORT_DIR,
    compute_eb_triage,
    discover_review_sources,
    export_eb_triage_products,
    inspect_candidate,
    load_reviewed_other_subset,
    resolve_local_paths,
)

# --- Notebook config -----------------------------------------------------
# Set REVIEW_SOURCE manually when the reviewed March 18 labels live in a
# specific DB or in an exported CSV/Parquet label table.
REVIEW_SOURCE = None
CANDIDATES_SOURCE = str(DEFAULT_CANDIDATES_SOURCE)
RUN_DIR = None
BUNDLE_DIR = None
EVENT_CLASS_FILTER = DEFAULT_EVENT_CLASS_FILTER
EXPORT_DIR = str(DEFAULT_EXPORT_DIR)

review_scan = discover_review_sources(event_class_filter=EVENT_CLASS_FILTER)
display(review_scan)

matching_sources = review_scan[review_scan["n_matching_reviews"] > 0].reset_index(drop=True)
if REVIEW_SOURCE is None:
    if matching_sources.empty:
        raise SystemExit(
            "No reviewed `other` labels were found in the standard review DB locations. "
            "Set REVIEW_SOURCE to the March 18 review DB or to an exported CSV/Parquet label table, "
            "then rerun this cell."
        )
    REVIEW_SOURCE = str(matching_sources.loc[0, "source_path"])
    display(Markdown(f"Using discovered review source: `{REVIEW_SOURCE}`"))

PATH_ROOT = BUNDLE_DIR or RUN_DIR
display(Markdown(f"Candidates source: `{CANDIDATES_SOURCE}`"))
display(Markdown(f"Path root for local light curves: `{PATH_ROOT}`" if PATH_ROOT else "Path root for local light curves: `None`"))


In [ ]:
other_subset = load_reviewed_other_subset(
    REVIEW_SOURCE,
    CANDIDATES_SOURCE,
    event_class_filter=EVENT_CLASS_FILTER,
)
other_subset = resolve_local_paths(other_subset, run_dir=PATH_ROOT)

if other_subset.empty:
    raise SystemExit(
        f"The selected review source does not contain any reviewed `{EVENT_CLASS_FILTER}` candidates."
    )

preview_cols = [
    col for col in [
        "candidate_id",
        "event_class",
        "status",
        "interest_score",
        "review_pass",
        "local_lightcurve_exists",
        "path",
        "local_lightcurve_path",
    ] if col in other_subset.columns
]
display(other_subset[preview_cols].head(10))
print(f"Loaded {len(other_subset):,} reviewed `{EVENT_CLASS_FILTER}` candidates.")
print("Localized path counts:", other_subset.attrs.get("localized_counts", {}))


In [ ]:
triage_df = compute_eb_triage(other_subset)
export_paths = export_eb_triage_products(other_subset, triage_df, export_dir=EXPORT_DIR)

display(pd.DataFrame({
    "artifact": list(export_paths.keys()),
    "path": [str(path) for path in export_paths.values()],
}))

triage_preview_cols = [
    col for col in [
        "candidate_id",
        "eb_likely_label",
        "eb_bin",
        "eb_score",
        "eb_score_notes",
        "stats_variability_lomb_scargle_best_period_days",
        "ls_fap_score",
        "dip_run_count",
        "spacing_cv",
        "known_eb_hint",
        "known_periodic_hint",
    ] if col in triage_df.columns
]
display(triage_df[triage_preview_cols].head(15))


In [ ]:
bin_counts = (
    triage_df["eb_bin"]
    .astype(str)
    .value_counts()
    .reindex(BIN_ORDER, fill_value=0)
    .rename("n_candidates")
    .to_frame()
)
display(bin_counts)

summary_cols = [
    col for col in [
        "eb_score",
        "stats_variability_lomb_scargle_best_period_days",
        "dip_run_count",
        "spacing_cv",
        "dipper_score",
    ] if col in triage_df.columns
]
summary = (
    triage_df.groupby("eb_bin", observed=False)[summary_cols]
    .agg(["count", "median"])
    .round(3)
)
display(summary)


In [ ]:
plot_df = triage_df.copy()
color_map = {
    "strong_eb_candidate": "#d62728",
    "possible_eb": "#ff7f0e",
    "maybe_periodic_not_eb": "#1f77b4",
    "unlikely_eb": "#7f7f7f",
}
colors = plot_df["eb_bin"].astype(str).map(color_map).fillna("#7f7f7f")

fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)

axes[0].scatter(plot_df["ls_fap_score"], plot_df["dip_run_count"], c=colors, s=16, alpha=0.75)
axes[0].set_xlabel("ls_fap_score = -log10(FAP)")
axes[0].set_ylabel("dip_run_count")
axes[0].set_title("Period significance vs repeat runs")
axes[0].grid(alpha=0.2)

axes[1].scatter(plot_df["spacing_cv"], plot_df["dip_amplitude_consistency"], c=colors, s=16, alpha=0.75)
axes[1].set_xlabel("spacing_cv")
axes[1].set_ylabel("dip_amplitude_consistency")
axes[1].set_title("Recurrence regularity vs amplitude repeatability")
axes[1].grid(alpha=0.2)

axes[2].scatter(plot_df["symmetry_abs"], plot_df["dip_duration_consistency"], c=colors, s=16, alpha=0.75)
axes[2].set_xlabel("symmetry_abs")
axes[2].set_ylabel("dip_duration_consistency")
axes[2].set_title("Symmetry vs duration repeatability")
axes[2].grid(alpha=0.2)

legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w", label=label, markerfacecolor=color, markersize=8)
    for label, color in color_map.items()
]
axes[0].legend(handles=legend_handles, title="eb_bin", loc="best")
plt.show()


In [ ]:
top_cols = [
    col for col in [
        "candidate_id",
        "eb_likely_label",
        "eb_bin",
        "eb_score",
        "eb_score_notes",
        "stats_variability_lomb_scargle_best_period_days",
        "stats_variability_lomb_scargle_peak_power",
        "stats_variability_lomb_scargle_fap",
        "ls_fap_score",
        "dip_run_count",
        "dipper_n_valid_dips",
        "spacing_cv",
        "dip_amplitude_consistency",
        "dip_duration_consistency",
        "symmetry_abs",
        "stats_variability_von_neumann_ratio",
        "stats_variability_stetson_J",
        "dipper_score",
        "known_eb_hint",
        "known_periodic_hint",
        "gaia_var_class",
        "gaia_eb_period",
        "gaia_eb_morph",
        "vsx_class",
        "catalog_match",
        "local_lightcurve_exists",
    ] if col in triage_df.columns
]

likelihood_summary = (
    triage_df.groupby("eb_likely_label", dropna=False)
    .agg(
        n_candidates=("candidate_id", "size"),
        n_known_eb_hint=("known_eb_hint", "sum"),
        n_known_periodic_hint=("known_periodic_hint", "sum"),
        median_eb_score=("eb_score", "median"),
    )
    .sort_index()
)

display(Markdown("## Likely vs not-likely EB summary"))
display(likelihood_summary)

if {"eb_likely_label", "known_eb_hint"}.issubset(triage_df.columns):
    display(Markdown("Known EB hints are only a proxy sanity check, not ground truth."))
    display(pd.crosstab(triage_df["eb_likely_label"], triage_df["known_eb_hint"], margins=True))

likely_df = triage_df.loc[triage_df["eb_likely_flag"]].copy()
not_likely_df = triage_df.loc[~triage_df["eb_likely_flag"]].copy()

display(Markdown("## Candidates classified as likely EBs"))
display(likely_df[top_cols].head(100))

not_likely_sort_cols = [
    col for col in ["known_eb_hint", "known_periodic_hint", "eb_score", "dipper_score", "candidate_id"]
    if col in not_likely_df.columns
]
if not_likely_sort_cols:
    not_likely_df = not_likely_df.sort_values(
        not_likely_sort_cols,
        ascending=[col == "candidate_id" for col in not_likely_sort_cols],
        na_position="last",
    )

display(Markdown("## Candidates classified as not likely EBs"))
display(not_likely_df[top_cols].head(100))


In [ ]:
def inspect(candidate_id: str):
    """Show metadata and, when possible, raw + phase-folded plots for one candidate."""
    return inspect_candidate(triage_df, candidate_id, run_dir=PATH_ROOT)


print("Use inspect(<candidate_id>) to examine a candidate in detail.")
print("Example:")
print("    inspect(triage_df.iloc[0]['candidate_id'])")
